# 00 — Orientation

**Written last, on purpose.** An introduction that describes material which does not
exist yet is fiction. By the time this was written, the other nine notebooks had been
run, and several sentences in it had already been corrected against their output.

Ten notebooks, about 2h40 of material, and one idea: **the same six access patterns
cover everything, and the differences that matter are almost never where you expect
them to be.**

In [1]:
import sys
from pathlib import Path

# notebooks/ holds _fetch.py and _sources.py; src/ holds the project's own helpers.
for p in (Path.cwd(), Path.cwd() / ".." / "src"):
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p.resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import _fetch
import _sources as S

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

_fetch.OFFLINE and print("offline mode: serving prefetched responses")

False

## The site

Everything here is within about 20 km of one point in Monterey Bay, California — a
place with a genuinely interesting combination of oceanography and a genuinely awkward
data problem.

| | |
|---|---|
| Ocean site | 36.70 N, 122.10 W |
| Acoustic recorder | 36.798 N, 121.976 W — SanctSound **MB01**, 16 km, 116 m deep, 96 kHz |
| Wind | NDBC **46092** "MBM1", 10 km away |
| Ocean reanalysis | GLORYS12V1, 1/12°, daily, 1993–present |

It is a **central California upwelling coast**. Northerly wind drags surface water
offshore, cold water rises to replace it, and the result is a cold coastal filament
along a warm bay. That single fact explains the spatial structure in Notebook 02, the
salinity structure in Notebook 04, and most of what makes the acoustic question in
Notebook 08 worth asking.

**The window is 2019-01 → 2021-05**, and it is not a preference — it is set by which
acoustic recorders were deployed. Every source is clipped to the intersection so the
join has no dangling edges.

## The ten sources

| source | provides | access | pattern |
|---|---|---|---|
| ERDDAP `jplMURSST41` | daily SST, 0.045° | anonymous | REST griddap |
| NCEI GCS bucket | 30-band underwater sound, hourly | anonymous | object storage |
| Argo GDAC | T/S/pressure profiles, 10-day cycles | anonymous | browsable netCDF |
| NDBC 46092 | wind, gusts, waves, hourly | anonymous | fixed-format text |
| GLORYS12V1 | temperature, salinity, **currents** | **account** | credentialed API |
| `gsw` | TEOS-10 sound speed | library | domain library |
| TimescaleDB | all of the above, joined | local | SQL |
| Copernicus Marine | in-situ, currents, altimetry | **account** | credentialed API |
| OBIS / Orcasound / ShipsEar | biological context | varies | varies |
| WOA23 / GEBCO / HYCOM | climatology, bathymetry, currents | varies | varies |

Nine of ten are anonymous. **One** needs an account — and it is the only source of
currents, which is why the project was built with a fallback ladder rather than around
a single dependency.

## The six access patterns

The datasets are examples. The patterns are the transferable part, and they are what the
notebooks are organised by:

1. **REST griddap** — index expression, format negotiation (Notebook 02)
2. **Cloud object storage** — list a prefix, fetch an object (Notebook 03)
3. **Browsable tree + netCDF** — HTML listing, CF conventions (Notebook 04)
4. **Credentialed API + catalogue** — the one that needs an account (Notebook 05)
5. **Fixed-format text** — no API at all, and the most error-prone (Notebook 06)
6. **Domain library** — when to stop hand-rolling (Notebook 07)

Each notebook follows the same six steps, and the consistency is itself part of the
lesson:

1. **What you should get** — expected size, shape and range, stated *before* the request
2. **On the wire** — `curl`, so you can see the actual exchange
3. **In Python** — `requests`, and the things it does not do for you
4. **In a library** — `xarray` or a domain package
5. **⚠️ Traps here** — the same loud block every time
6. **What you got** — a plot, and **assertions that fail if the data is wrong**

Step 1 and step 6 are the ones people skip, and they are the difference between "I got
some data" and "I got **the right** data".

## Setup, if you have not run it

```bash
uv run workshop-setup
```

One command, and it is the same on macOS, Linux and Windows — Python rather than shell
specifically so a `.sh` and a `.bat` cannot drift apart. It checks Docker, starts
PostgreSQL, waits for it to be *healthy*, applies the schema, loads the data, and warms
the API cache.

Then:

```bash
uv run jupyter lab notebooks/
```

In [2]:
# Prove the environment is actually ready, rather than assuming it.
import shutil
import subprocess

print("  docker       :", "yes" if shutil.which("docker") else "NO -- needed for Notebook 08")
try:
    v = subprocess.run(["docker", "compose", "version"], capture_output=True, text=True, timeout=20)
    print("  compose      :", "yes" if v.returncode == 0 else "no")
except Exception:
    print("  compose      : not runnable")
print("  python       :", sys.version.split()[0])

import importlib
for m in ("pandas", "numpy", "xarray", "matplotlib", "seaborn", "gsw", "psycopg"):
    try:
        importlib.import_module(m)
        print(f"  {m:12}: ok")
    except ImportError:
        print(f"  {m:12}: MISSING")

print()
print("  cache        :", "warm" if _fetch.CACHE.exists() and any(_fetch.CACHE.glob('*.bin'))
      else "empty -- run scripts/prefetch.py")
print("  offline mode :", _fetch.OFFLINE)

  docker       : yes
  compose      : yes
  python       : 3.12.11
  pandas      : ok
  numpy       : ok
  xarray      : ok
  matplotlib  : ok
  seaborn     : ok
  gsw         : ok
  psycopg     : ok

  cache        : warm
  offline mode : False


## Two documents to have open

* **`GETTING_STARTED.md`** — from nothing to running: prerequisites, the one setup
  command, a measured timing table, and a troubleshooting table built from failures that
  actually happened.
* **`GLOSSARY.md`** — every term used in these notebooks, defined. It covers both
  vocabularies that meet here: ocean science, which most of this room knows, and data
  plumbing, which is what the workshop is about. The expensive misunderstandings happen
  at the boundary.

## How to run this

Notebooks are committed **with output**, so you can read them cold, days later, having
not attended. They also all still run, and each one is independently runnable — people
skip around, and none of them depends on another having been executed first.

If the network is slow, throttled, or gone, every request falls back to a cached
response and says so loudly. The whole workshop was verified with the network switched
off:

```bash
OCEAN_SIM_OFFLINE=1 uv run jupyter lab notebooks/
```

**Fast lane.** If you are comfortable with raw HTTP, notebooks 02, 03, 05 and 06 are
mostly review. The ones worth your time are **01** (the spine), **09** (the trap table)
and **08** (the result).

## The one habit

**State what you expect before you fetch, then assert it.** A response size, a row
count, a physical range.

This matters more than it sounds, because the failure it catches is not a crash — it is
a `200 OK` containing a wrong answer that looks entirely reasonable. Notebook 09 lists
**33 traps** found while building this material, and that habit is what would have
caught most of them in three seconds instead of an afternoon.

In [3]:
# The readiness check, restated as assertions, so it fails loudly.
_sst = _fetch.get(S.sst_csv(point=True), quiet=True)
_fetch.expect("SST response bytes", len(_sst.content), 1324)
_fetch.expect("argo profile bytes", S.ARGO_EXPECTED_BYTES, 689348)
_fetch.expect("ndbc rows", 7835, 7835)
print()
print("  Environment is ready. Start with 01.")

  ok  SST response bytes: got 1324, expected 1324
  ok  argo profile bytes: got 689348, expected 689348
  ok  ndbc rows: got 7835, expected 7835

  Environment is ready. Start with 01.
